In [ ]:
import polars as pl
from pathlib import Path

In [ ]:
base_path = Path("data/t_ecd_small_partial/dataset/small")

retail_events_path = base_path / "retail" / "events"
lf_retail = pl.scan_parquet(str(retail_events_path / "*.pq"))

print("✅ RETAIL EVENTS:")
print("Схема:", lf_retail.collect_schema())
print("\nПервые 5 строк:")
print(lf_retail.head(5).collect())



In [ ]:
marketplace_events_path = base_path / "marketplace" / "events"
lf_marketplace = pl.scan_parquet(str(marketplace_events_path / "*.pq"))

print("\n✅ MARKETPLACE EVENTS:")
print("Схема:", lf_marketplace.collect_schema())
print("\nПервые 5 строк:")
print(lf_marketplace.head(5).collect())


✅ MARKETPLACE EVENTS:
Схема: Schema({'timestamp': Duration(time_unit='us'), 'user_id': UInt64, 'item_id': String, 'subdomain': String, 'action_type': String, 'os': String})

Первые 5 строк:
shape: (5, 6)
┌───────────────────┬──────────┬────────────────┬───────────┬─────────────┬─────────┐
│ timestamp         ┆ user_id  ┆ item_id        ┆ subdomain ┆ action_type ┆ os      │
│ ---               ┆ ---      ┆ ---            ┆ ---       ┆ ---         ┆ ---     │
│ duration[μs]      ┆ u64      ┆ str            ┆ str       ┆ str         ┆ str     │
╞═══════════════════╪══════════╪════════════════╪═══════════╪═════════════╪═════════╡
│ 1200d 222299µs    ┆ 20322221 ┆ nfmcg_18961369 ┆ u2i       ┆ view        ┆ ios     │
│ 1200d 422277µs    ┆ 26590935 ┆ nfmcg_10940040 ┆ search    ┆ view        ┆ ios     │
│ 1200d 815192µs    ┆ 25438056 ┆ nfmcg_3329926  ┆ other     ┆ view        ┆ android │
│ 1200d 1s 243709µs ┆ 12366330 ┆ nfmcg_28252252 ┆ catalog   ┆ view        ┆ ios     │
│ 1200d 1s 582237µs ┆

In [ ]:


# ===== 3. Базовая статистика (безопасно!) =====
print("\n📊 СТАТИСТИКА RETAIL:")
stats_retail = (
    lf_retail
    .select([
        pl.len().alias("total_rows"),
        pl.col("user_id").n_unique().alias("unique_users"),
        pl.col("item_id").n_unique().alias("unique_items"),
        pl.col("brand_id").n_unique().alias("unique_brands"),
    ])
    .collect()
)
print(stats_retail)

print("\n📊 СТАТИСТИКА MARKETPLACE:")
stats_mp = (
    lf_marketplace
    .select([
        pl.len().alias("total_rows"),
        pl.col("user_id").n_unique().alias("unique_users"),
        pl.col("item_id").n_unique().alias("unique_items"),
        pl.col("brand_id").n_unique().alias("unique_brands"),
    ])
    .collect()
)
print(stats_mp)

# ===== 4. Action types по доменам =====
print("\n🔥 Action types:")
for domain, lf in [("Retail", lf_retail), ("Marketplace", lf_marketplace)]:
    actions = (
        lf
        .groupby("action_type")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )
    print(f"{domain}:")
    print(actions)
